In [ ]:
import os
import openai
from openai import OpenAI
from dotenv import load_dotenv

class AsistenteMetro:
    def __init__(self):
        load_dotenv()
        self.client = OpenAI(
            base_url=os.environ.get("GITHUB_BASE_URL"),
            api_key=os.environ.get("GITHUB_TOKEN")
        )
        # Límite de memoria para no gastar de más
        self.limite_mensajes = 6 
        self.memoria = []
        self._inicializar_memoria()

    def _inicializar_memoria(self):
        contexto_red = """
        INFORMACIÓN DE TARIFAS (General):
        - Horario Punta (07:00-08:59 y 18:00-19:59): $840
        - Horario Valle (09:00-17:59 y 20:00-20:44): $760
        - Horario Bajo (06:00-06:59 y 20:45-23:00): $680
        - Tarifa Estudiante (TNE) y Adulto Mayor: $240 en todo horario.

        ESTRUCTURA DE LÍNEAS Y COMBINACIONES CLAVE:
        - Línea 1 (Roja): Terminales San Pablo / Los Dominicos. Combina con L2, L3, L4 (Tobalaba), L5 y L6.
        - Línea 4 (Azul): Terminales Tobalaba / Plaza Puente Alto. Combina con L1 (Tobalaba), L3 y L5.
        """
        horario_actual = "Día hábil, 18:30 hrs (Horario Punta)"
        alerta_operativa = "Línea 1 presenta retrasos por asistencia a pasajero en estación Baquedano."

        prompt_sistema = f"""
        Actúa como el asistente virtual oficial de servicio al cliente de Metro de Santiago.
        Contexto de la red: {contexto_red}
        Horario actual: {horario_actual}
        Alertas operativas: {alerta_operativa}

        Reglas:
        1. Indica la línea inicial, combinaciones y estación final.
        2. Informa la tarifa exacta del viaje.
        3. Advierte sobre alertas operativas.
        """
        self.memoria = [{"role": "system", "content": prompt_sistema}]

    def _gestionar_tokens(self):
        if len(self.memoria) > self.limite_mensajes + 1:
            self.memoria = [self.memoria[0]] + self.memoria[-self.limite_mensajes:]

    def consultar_ruta_streaming(self, pregunta_usuario):
        self.memoria.append({"role": "user", "content": pregunta_usuario})
        self._gestionar_tokens()

        try:
            # Petición a la API con streaming activado
            response = self.client.chat.completions.create(
                model="gpt-4o",
                messages=self.memoria,
                temperature=0.7,
                max_tokens=600,
                stream=True 
            )
            
            print("\nAsistente: ", end="")
            respuesta_completa = ""
            
            # Ciclo for para imprimir palabra por palabra
            for chunk in response:
                # Este if evita errores si el pedacito de texto viene vacío
                if chunk.choices and chunk.choices[0].delta and chunk.choices[0].delta.content is not None:
                    texto = chunk.choices[0].delta.content
                    print(texto, end="", flush=True)
                    respuesta_completa += texto
            
            print("\n") # Salto de línea final
            
            # Guardar la respuesta en memoria
            self.memoria.append({"role": "assistant", "content": respuesta_completa})

        except openai.RateLimitError:
            print("\nAsistente: ⚠️ El sistema está recibiendo muchas consultas. Por favor, intenta en unos segundos.\n")
            self.memoria.pop() # Borramos la pregunta de la memoria porque falló la conexión
        except openai.APIConnectionError:
            print("\nAsistente: 🔌 Hay un problema conectando con el servidor. Revisa tu internet.\n")
            self.memoria.pop()
        except Exception as e:
            print(f"\nAsistente: ❌ Ocurrió un error: {str(e)}\n")
            self.memoria.pop()

# ==========================================
# Ejecución principal
# ==========================================
if __name__ == "__main__":
    asistente = AsistenteMetro()
    print("🚇 Asistente de Metro iniciado (Escribe 'salir' para terminar)\n")

    while True:
        texto_usuario = input("Tú: ")
        if texto_usuario.lower() == 'salir':
            print("Asistente: ¡Buen viaje! Cerrando sistema.")
            break
            
        asistente.consultar_ruta_streaming(texto_usuario)